# Week 1 — SELECT, WHERE, ORDER BY, LIMIT: Filtered Exploration
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises (Thursday, Group Work)**

Wednesday you practiced `SELECT`, `WHERE`, `AND`/`OR`, and `ORDER BY`/`LIMIT` one clause at a time. Today's group exercise asks you to combine them to answer five independent business questions against the real Olist tables — no scaffolding beyond the question itself.

Each question below is **three cells**:

1. A markdown cell with the task and an **Expected** result (where the curriculum gives a fixed number).
2. A blank `%%sql qN <<` answer cell — replace `-- Your query here` with your query, aliasing columns to match what the check cell expects.
3. A locked check cell (plain Python) — run it right after your query. A ✅ print means your query is correct.

Fill in each `%%sql` cell, then run the check cell below it — a ✅ means your query is correct. Do not edit the check cells. Run the setup cell first, then work top to bottom. Work in your group; verify answers together before moving to the next question.

### Setup — run this first
Loads the eight Olist tables into a file-based SQLite database at `/content/olist.db` and connects the `%%sql` cell magic. Because `autopandas` is on, every `%%sql` result comes back as a pandas DataFrame, which is exactly what the check cells inspect below.

In [ ]:
"""
Standard Olist SQL Setup for Google Colab — Phase 2b SQL

Use this block as the FIRST code cell in every Phase 2b notebook (all weeks).

Design notes (why this differs from the Phase 2a pandas setup):
- We teach SQL with the `%%sql` cell magic (jupysql) so beginners write raw
  SQL, not `pd.read_sql("...")` strings. Students meet SQL syntax first;
  the Python wrapper comes later (Phase 2c).
- jupysql opens its OWN database connection. A `sqlite3.connect(":memory:")`
  database is private to that one connection, so a separate `%sql sqlite://`
  connection would see ZERO tables. We therefore build a **file-based**
  SQLite database at /content/olist.db and point both pandas (for loading)
  and jupysql (for querying) at the same file.
- `autopandas = True` makes every `%%sql` cell return a pandas DataFrame, so
  self-check cells can `assert` on `.iloc`/`.shape` directly.

After this cell runs, query with:

    %%sql
    SELECT * FROM orders LIMIT 5

Or capture a result for checking:

    %%sql result <<
    SELECT COUNT(*) AS n FROM orders WHERE order_status = 'delivered'
    # then in a plain Python cell:  assert int(result.iloc[0]['n']) == 96478
"""

import os
import sqlite3
import pandas as pd
from google.colab import drive

# 1) Mount Drive and locate the Olist CSVs
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/cohort7/datasets/olist"

# 2) Build a file-based SQLite database (shared by pandas + jupysql)
DB_PATH = "/content/olist.db"
conn = sqlite3.connect(DB_PATH)

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")

conn.close()
print("\nDatabase ready.")

# 3) Connect jupysql to the SAME database file
%load_ext sql
%config SqlMagic.autopandas = True          # %%sql cells return DataFrames
%config SqlMagic.feedback = 0               # quieter output
%sql sqlite:////content/olist.db

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Question 1 — Orders with a NULL delivery date

Not every order that's placed ends up with a recorded delivery date. Write a query that counts how many rows in `orders` have `order_delivered_customer_date` equal to `NULL` — remember, you can never test for NULL with `=`. Alias your count column as `null_delivery`.

**Expected:** 2,965 orders with a NULL delivery date

In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
assert int(q1.iloc[0]['null_delivery']) == 2965, "Q1: expected 2,965 orders with a NULL delivery date"
print("✅ Q1 correct")

## Question 2 — The 5 most expensive individual items

The `order_items` table holds one row per item within an order, with a `price` for that specific item. Write a query that lists the `order_id`, `product_id`, and `price` of the 5 most expensive individual items, sorted from most to least expensive.

**Expected:** exactly 5 rows, sorted by `price` from highest to lowest (no single fixed value — this is open exploration).

In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
assert len(q2) == 5, "Q2: expected exactly 5 rows"
assert q2['price'].is_monotonic_decreasing, "Q2: expected price sorted highest to lowest (ORDER BY price DESC)"
print("✅ Q2 correct")

## Question 3 — Payments made by boleto

Brazilian shoppers can pay several ways, including the `boleto` bank-slip method. Write a query that counts how many rows in `order_payments` have `payment_type` equal to `boleto`. Alias your count column as `boleto_count`.

**Expected:** 19,784 boleto payments

In [ ]:
%%sql q3 <<
-- Your query here

In [ ]:
# --- CHECK Q3 — do not edit ---
assert int(q3.iloc[0]['boleto_count']) == 19784, "Q3: expected 19,784 boleto payments"
print("✅ Q3 correct")

## Question 4 — Customers in the state of SP

The `customers` table has a `customer_state` column with two-letter Brazilian state codes. Write a query that counts how many rows have `customer_state` equal to `SP`. Alias your count column as `sp_customers`.

**Expected:** 41,746 customers in SP

In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
assert int(q4.iloc[0]['sp_customers']) == 41746, "Q4: expected 41,746 customers in SP"
print("✅ Q4 correct")

## Question 5 — The first 10 worst reviews

The `order_reviews` table has a `review_score` column from 1 (worst) to 5 (best). Write a query that lists the first 10 rows where `review_score` equals 1, showing `order_id`, `review_score`, and `review_comment_message`. Which column holds the customer's written comment?

**Expected:** exactly 10 rows, every one with `review_score` equal to 1. (The customer comment lives in `review_comment_message`.)

In [ ]:
%%sql q5 <<
-- Your query here

In [ ]:
# --- CHECK Q5 — do not edit ---
assert len(q5) == 10, "Q5: expected exactly 10 rows"
assert (q5['review_score'] == 1).all(), "Q5: expected every row to have review_score == 1"
print("✅ Q5 correct")